# 02 — Benchmarking de Modelos, Calibración & Explicabilidad
**Credit Policy Optimizer** — Modelado Estadístico y Calibración de Probabilidades

### Objetivos del Experimento:
1. Comparar familias de modelos: Regresión Logística (Scorecard estándar), Random Forest y LightGBM.
2. Evaluar métricas regulatorias bancarias: **ROC-AUC**, **PR-AUC**, **Coeficiente de Gini ($2 \cdot \text{AUC} - 1$)** y estadístico **Kolmogorov-Smirnov (KS)**.
3. Analizar el impacto de la **Calibración Isotónica** vs. **Sigmoide** (Platt Scaling) mediante curvas de confiabilidad, Brier Score y ECE.
4. Explorar interpretabilidad global y razones adversas (*adverse action reasons*).


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

from credit_policy_optimizer.data.generator import PortfolioSimulator
from credit_policy_optimizer.models.pipeline import (
    LightGBMConfig,
    build_credit_pipeline,
    DEFAULT_NUMERIC_FEATURES,
    DEFAULT_CATEGORICAL_FEATURES
)
from credit_policy_optimizer.models.calibration import (
    calibrate_pipeline,
    compute_expected_calibration_error,
    compute_gini_coefficient,
    compute_ks_statistic,
    evaluate_calibration
)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)


## 1. Carga y Partición Estratificada de Datos


In [ ]:
sim = PortfolioSimulator(seed=42)
df = sim.simulate(n_samples=30_000)

features = DEFAULT_NUMERIC_FEATURES + DEFAULT_CATEGORICAL_FEATURES
X = df.select(features)
y = df["default_flag"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_train, X_cal, y_train, y_cal = train_test_split(
    X_train, y_train, test_size=0.20, random_state=42, stratify=y_train
)

print(f"Muestra de Entrenamiento: {X_train.height:,}")
print(f"Muestra de Calibración:  {X_cal.height:,}")
print(f"Muestra de Prueba (OOS): {X_test.height:,}")


## 2. Entrenamiento del Benchmark de Modelos
Entrenamos:
- **Baseline**: Regresión Logística (Scorecard lineal).
- **Ensamble Bagging**: Random Forest.
- **Ensamble Boosting**: LightGBM Classifier.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer(
    transformers=[("num", StandardScaler(), DEFAULT_NUMERIC_FEATURES)],
    remainder="drop"
)

# 1. Logistic Regression
lr_pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(random_state=42, max_iter=500))
])
lr_pipeline.fit(X_train.to_pandas(), y_train)

# 2. Random Forest
rf_pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1))
])
rf_pipeline.fit(X_train.to_pandas(), y_train)

# 3. LightGBM (Pipeline Oficial del Proyecto)
lgb_pipeline = build_credit_pipeline(model_config=LightGBMConfig(random_state=42))
lgb_pipeline.fit(X_train, y_train)

print("Todos los modelos han sido entrenados exitosamente.")


## 3. Comparativa de Métricas de Discriminación Bancaria
Evaluamos ROC-AUC, PR-AUC, Gini y KS sobre el conjunto de prueba (Out-Of-Sample).


In [ ]:
models = {
    "Logistic Regression": lr_pipeline,
    "Random Forest": rf_pipeline,
    "LightGBM": lgb_pipeline
}

metrics_list = []
preds_dict = {}

for name, model in models.items():
    if name == "LightGBM":
        probs = model.predict_proba(X_test)[:, 1]
    else:
        probs = model.predict_proba(X_test.to_pandas())[:, 1]
        
    preds_dict[name] = probs
    
    auc = roc_auc_score(y_test, probs)
    pr_auc = average_precision_score(y_test, probs)
    gini = compute_gini_coefficient(y_test, probs)
    ks_stat, ks_thresh = compute_ks_statistic(y_test, probs)
    
    metrics_list.append({
        "Modelo": name,
        "ROC-AUC": round(auc, 4),
        "PR-AUC": round(pr_auc, 4),
        "Gini": round(gini, 4),
        "KS Statistic": round(ks_stat, 4),
        "Umbral KS": round(ks_thresh, 4)
    })

comparison_df = pl.DataFrame(metrics_list)
comparison_df


### Curvas ROC y Curvas de Separación Kolmogorov-Smirnov (KS)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Curva ROC
for name, probs in preds_dict.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, probs):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.6)
axes[0].set_title("Curvas ROC Comparativas")
axes[0].set_xlabel("Tasa de Falsos Positivos (FPR)")
axes[0].set_ylabel("Tasa de Verdaderos Positivos (TPR)")
axes[0].legend()

# Curva KS para LightGBM
lgb_probs = preds_dict["LightGBM"]
fpr, tpr, thresholds = roc_curve(y_test, lgb_probs)
ks_stat, ks_thresh = compute_ks_statistic(y_test, lgb_probs)

axes[1].plot(thresholds, tpr, label="TPR (Bads Acumulados)", color="crimson")
axes[1].plot(thresholds, fpr, label="FPR (Goods Falsos)", color="navy")
axes[1].plot(thresholds, tpr - fpr, label=f"Diferencia KS (Máx = {ks_stat:.3f} en p={ks_thresh})", color="forestgreen", linestyle="--")
axes[1].set_xlim([0, 1])
axes[1].set_title("Gráfico de Separación Kolmogorov-Smirnov (LightGBM)")
axes[1].set_xlabel("Umbral de Probabilidad de Corte")
axes[1].set_ylabel("Tasa Acumulada")
axes[1].legend()

plt.tight_layout()
plt.show()


## 4. Diagnóstico y Calibración de Probabilidades
¿Por qué calibrar? Los modelos basados en árboles no están entrenados bajo log-loss pura y suelen distorsionar los extremos probabilísticos. Para la toma de decisiones económicas ($EV$), la probabilidad de default predicha debe coincidir con la frecuencia observada de impagos.


In [ ]:
# Calibración Isotónica y Sigmoide sobre LightGBM
calibrated_isotonic = calibrate_pipeline(lgb_pipeline, X_cal, y_cal, method="isotonic")
calibrated_sigmoid = calibrate_pipeline(lgb_pipeline, X_cal, y_cal, method="sigmoid")

raw_probs = lgb_pipeline.predict_proba(X_test)[:, 1]
iso_probs = calibrated_isotonic.predict_proba(X_test)[:, 1]
sig_probs = calibrated_sigmoid.predict_proba(X_test)[:, 1]

ece_raw, _, _, _, _ = compute_expected_calibration_error(y_test, raw_probs)
ece_iso, _, acc_iso, conf_iso, counts_iso = compute_expected_calibration_error(y_test, iso_probs)
ece_sig, _, _, _, _ = compute_expected_calibration_error(y_test, sig_probs)

print(f"ECE Modelo Crudo (LightGBM):    {ece_raw:.4f}")
print(f"ECE Calibración Sigmoide:       {ece_sig:.4f}")
print(f"ECE Calibración Isotónica:      {ece_iso:.4f} (Reducción: {((ece_raw - ece_iso)/ece_raw)*100:.1f}%)")


### Curva de Confiabilidad (Reliability Diagram)


In [ ]:
from sklearn.calibration import calibration_curve

prob_true_raw, prob_pred_raw = calibration_curve(y_test, raw_probs, n_bins=10)
prob_true_iso, prob_pred_iso = calibration_curve(y_test, iso_probs, n_bins=10)

plt.figure(figsize=(8, 6))
plt.plot(prob_pred_raw, prob_true_raw, "s-", label=f"LightGBM Crudo (ECE={ece_raw:.3f})", color="salmon")
plt.plot(prob_pred_iso, prob_true_iso, "o-", label=f"LightGBM Isotónico (ECE={ece_iso:.3f})", color="navy")
plt.plot([0, 1], [0, 1], "k--", label="Calibración Perfecta")
plt.title("Diagrama de Confiabilidad (Curva de Calibración)")
plt.xlabel("Probabilidad Media Predicha")
plt.ylabel("Fracción Empírica de Impagos (Default Rate)")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Explicabilidad Global de Factores de Riesgo
Inspeccionamos la importancia relativa de variables del modelo LightGBM final.


In [ ]:
lgb_estimator = lgb_pipeline.named_steps["classifier"]
prep = lgb_pipeline.named_steps["preprocessor"]
try:
    feature_names = [f.replace("num__", "").replace("cat__", "") for f in prep.get_feature_names_out()]
except Exception:
    feature_names = [f"feat_{i}" for i in range(len(lgb_estimator.feature_importances_))]

importances = lgb_estimator.feature_importances_

fi_df = pl.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort("Importance", descending=True)

plt.figure(figsize=(10, 5))
sns.barplot(x="Importance", y="Feature", data=fi_df.to_pandas(), palette="mako")
plt.title("Importancia de Features en LightGBM (Split Count)")
plt.tight_layout()
plt.show()


## 6. Conclusiones del Benchmark
1. **Superioridad de LightGBM**: Supera a la Regresión Logística y a Random Forest en AUC, Gini y KS.
2. **Impacto de la Calibración**: La regresión isotónica reduce el ECE drásticamente, garantizando que las probabilidades sean estimaciones no sesgadas de la tasa de default real.
3. El modelo calibrado isotónico queda validado como base para el motor económico de políticas.
